# Import Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
    PowerTransformer
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.model_selection import cross_val_score
import joblib

# Load Dataset

In [2]:
df = pd.read_csv(r"C:\Users\HP\Downloads\World-happiness-report-2024.csv")
df.tail()

,Country name,Regional indicator,Ladder score,upperwhisker,lowerwhisker,Log GDP per capita,Social support,Healthy life expectancy,Freedom to make life choices,Generosity,Perceptions of corruption,Dystopia + residual
138,Congo (Kinshasa),Sub-Saharan Africa,3.295,3.462,3.128,0.534,0.665,0.262,0.473,0.189,0.072,1.102
139,Sierra Leone,Sub-Saharan Africa,3.245,3.366,3.124,0.654,0.566,0.253,0.469,0.181,0.053,1.068
140,Lesotho,Sub-Saharan Africa,3.186,3.469,2.904,0.771,0.851,0.000,0.523,0.082,0.085,0.875
141,Lebanon,Middle East and North Africa,2.707,2.797,2.616,1.377,0.577,0.556,0.173,0.068,0.029,-0.073
142,Afghanistan,South Asia,1.721,1.775,1.667,0.628,0.000,0.242,0.000,0.091,0.088,0.672


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 143 entries, 0 to 142
Data columns (total 12 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Country name                  143 non-null    object 
 1   Regional indicator            143 non-null    object 
 2   Ladder score                  143 non-null    float64
 3   upperwhisker                  143 non-null    float64
 4   lowerwhisker                  143 non-null    float64
 5   Log GDP per capita            140 non-null    float64
 6   Social support                140 non-null    float64
 7   Healthy life expectancy       140 non-null    float64
 8   Freedom to make life choices  140 non-null    float64
 9   Generosity                    140 non-null    float64
 10  Perceptions of corruption     140 non-null    float64
 11  Dystopia + residual           140 non-null    float64
dtypes: float64(10), object(2)
memory usage: 13.5+ KB


In [4]:
df.columns

Index(['Country name', 'Regional indicator', 'Ladder score', 'upperwhisker',
       'lowerwhisker', 'Log GDP per capita', 'Social support',
       'Healthy life expectancy', 'Freedom to make life choices', 'Generosity',
       'Perceptions of corruption', 'Dystopia + residual'],
      dtype='object')

# Remove Unnecessary rows/columns

In [5]:
df = df.dropna()
df = df.drop_duplicates()
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 140 entries, 0 to 142
Data columns (total 12 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Country name                  140 non-null    object 
 1   Regional indicator            140 non-null    object 
 2   Ladder score                  140 non-null    float64
 3   upperwhisker                  140 non-null    float64
 4   lowerwhisker                  140 non-null    float64
 5   Log GDP per capita            140 non-null    float64
 6   Social support                140 non-null    float64
 7   Healthy life expectancy       140 non-null    float64
 8   Freedom to make life choices  140 non-null    float64
 9   Generosity                    140 non-null    float64
 10  Perceptions of corruption     140 non-null    float64
 11  Dystopia + residual           140 non-null    float64
dtypes: float64(10), object(2)
memory usage: 14.2+ KB


# Select Features and Target

In [6]:
X = df.drop(columns=["Country name","Ladder score","upperwhisker","lowerwhisker","Dystopia + residual"])
y = df["Ladder score"]

In [7]:
X.head()

,Regional indicator,Log GDP per capita,Social support,Healthy life expectancy,Freedom to make life choices,Generosity,Perceptions of corruption
0,Western Europe,1.844,1.572,0.695,0.859,0.142,0.546
1,Western Europe,1.908,1.520,0.699,0.823,0.204,0.548
2,Western Europe,1.881,1.617,0.718,0.819,0.258,0.182
3,Western Europe,1.878,1.501,0.724,0.838,0.221,0.524
4,Middle East and North Africa,1.803,1.513,0.740,0.641,0.153,0.193


In [8]:
X.columns

Index(['Regional indicator', 'Log GDP per capita', 'Social support',
       'Healthy life expectancy', 'Freedom to make life choices', 'Generosity',
       'Perceptions of corruption'],
      dtype='object')

# Train-Test Split

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.25,random_state=42)

# Define Preprocessing Pipelines

In [10]:
numeric_features = ["Log GDP per capita","Social support","Healthy life expectancy",
                    "Freedom to make life choices","Generosity","Perceptions of corruption"]
categorical_features = ["Regional indicator"]
num_pipeline = Pipeline([
    ("scaler",StandardScaler()),
    ("power",PowerTransformer(method="yeo-johnson"))
])
cat_pipeline = Pipeline([
    ("encoder",OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("num",num_pipeline,numeric_features),
    ("cat",cat_pipeline,categorical_features)
])

# Create full pipeline (Preprocessing + Model)

In [11]:
pipeline = Pipeline([
    ("preprocessing",preprocessor),
    ("model",LinearRegression())
])

# Train the Pipeline

In [12]:
pipeline.fit(X_train,y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler()),
                                                                  ('power',
                                                                   PowerTransformer())]),
                                                  ['Log GDP per capita',
                                                   'Social support',
                                                   'Healthy life expectancy',
                                                   'Freedom to make life '
                                                   'choices',
                                                   'Generosity',
                                                   'Perceptions of '
                                                   'corruption']),
                                                 ('cat',
                                                  Pipeline(steps=[('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Regional indicator'])])),
                ('model', LinearRegression())])

# Evaluate Model

In [13]:
y_train_pred = pipeline.predict(X_train)
print("Training R2 Score:",r2_score(y_train,y_train_pred)*100)
print("Training MAE Score:",mean_absolute_error(y_train,y_train_pred))
rmse = np.sqrt(mean_squared_error(y_train,y_train_pred))
print("Training RMSE Score:",rmse)

Training R2 Score: 86.75701133629323
Training MAE Score: 0.3251773704228751
Training RMSE Score: 0.42514072264602537


In [14]:
y_pred = pipeline.predict(X_test)
print("R2 Score:",r2_score(y_test,y_pred)*100)
print("MAE Score:",mean_absolute_error(y_test,y_pred))
rmse = np.sqrt(mean_squared_error(y_test,y_pred))
print("RMSE Score:",rmse)

R2 Score: 78.24977242383633
MAE Score: 0.4487300991591412
RMSE Score: 0.5597641301463528


In [15]:
from sklearn.model_selection import KFold
kf = KFold(n_splits=5,shuffle=True,random_state=42)
scores = cross_val_score(pipeline,X,y,cv=kf,scoring='r2')
print(scores)
print("Mean R2:", scores.mean())
print("Std:", scores.std())

[0.78725096 0.77084696 0.7536041  0.77715441 0.86933269]
Mean R2: 0.7916378221248686
Std: 0.04035921514847424


# Save Full Pipeline (Deployment Ready)

In [16]:
import pickle
with open("happiness_pipeline.pkl","wb") as f:
    pickle.dump(pipeline,f)

In [17]:
joblib.dump(pipeline,"happiness_pipeline.pkl")
print("pipeline saved in pkl file")

pipeline saved in pkl file


# Load Pipeline and Predict (Deployment Simulation)

In [18]:
loaded_pipeline = joblib.load("happiness_pipeline.pkl")
new_country = pd.DataFrame({
    "Regional indicator":["Western Europe"],
    "Log GDP per capita":[1.878],
    "Social support":[1.501],
    "Healthy life expectancy":[0.724],
    "Freedom to make life choices":[0.838],
    "Generosity":[0.221],
    "Perceptions of corruption":[0.524]
})
prediction = loaded_pipeline.predict(new_country)
print("Ladder Score:",prediction[0])

Ladder Score: 7.483953547649792
